In [5]:
%reset -f

import os
import torch
import torch.nn as nn
import random
import matplotlib.pyplot as plt
from PIL import Image
import torchvision
from torchvision.transforms.functional import InterpolationMode
from torchvision.io.image import ImageReadMode
from tqdm import tqdm
import json
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split
import gc
from typing import Any, List, Dict
from transformers import AutoTokenizer, AutoModel
from diffusers import AutoencoderKL, FluxTransformer2DModel
from peft import LoraConfig, get_peft_model

torch.cuda.empty_cache()

In [6]:
class PokemonDataset(Dataset):
    def __init__(self, data: torch.Tensor, captions: list[str]):
        assert data.shape[0] == len(captions)
        self.data:torch.Tensor = data
        self.captions = captions

    def __len__(self) -> int:
        return self.data.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, str]:
        return (self.data[idx], self.captions[idx])

In [7]:
def resize_dataset(raw_dir: str = "./images/raw", target_dir: str = "./images/1024x1024") -> None:
    if os.path.exists(target_dir):
        os.rmdir(target_dir)
    os.makedirs(target_dir, exist_ok=True)

    valid_extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
    files = [f for f in os.listdir(raw_dir) if f.lower().endswith(valid_extensions)]

    trans_resize = torchvision.transforms.Resize((1024, 1024), InterpolationMode.BICUBIC, antialias=True)

    for filename in tqdm(files, desc="Resizing images"):
        img_path: str = os.path.join(raw_dir, filename)
        raw_image = torchvision.io.read_image(img_path, ImageReadMode.RGB_ALPHA)
        
        alpha = raw_image[3:4, :, :].float() / 255.0
        rgb = raw_image[:3, :, :].float()
        white_bg = torch.ones_like(rgb) * 255.0
        
        flat_image = (rgb * alpha + white_bg * (1.0 - alpha)).to(torch.uint8)

        output_image: torch.Tensor = trans_resize(flat_image).cpu() 
        torchvision.io.write_png(output_image, os.path.join(target_dir, filename), 0)

def load_dataset() -> PokemonDataset:
    images_dir:str = os.path.join(os.getcwd(), "images", "1024x1024")
    images_path:list[str] = os.listdir(images_dir)
    images:list[torch.Tensor] = list[torch.Tensor]()
    captions: list[str] = []

    captions_dict: dict[str, str] = {}
    with open("./captions.json", encoding="utf-8") as f:
        captions_dict = json.loads(f.read())

    for image_local_path in images_path:
        img_name: str = image_local_path.removesuffix(".png")
        if img_name not in captions_dict:
            print(f"{img_name} doesn't have caption!")
            continue

        image_absolute_path:str = os.path.join(images_dir, image_local_path)
        image:torch.Tensor = torchvision.io.read_image(image_absolute_path, ImageReadMode.RGB).type(torch.float32) / 255.0
        image = image * 2.0 - 1.0
        images.append(image)
        captions.append(captions_dict[img_name])

    return PokemonDataset(torch.stack(images).cpu(), captions)

def test_dataset() -> None:
    images = load_dataset()
    idx = random.randint(0, len(images) - 1)
    img_tensor, _ = images[idx]

    img_to_show = (img_tensor * 0.5 + 0.5).permute(1, 2, 0).numpy()

    plt.imshow(img_to_show)
    plt.title(f"Random Pokémon Sample (Index: {idx})")
    plt.axis("off")
    plt.show()

In [ ]:
def train_pokemon_flux_lora(
    dataset: PokemonDataset,
    vae_id: str = "black-forest-labs/flux-2-vae",
    text_encoder_id: str = "Qwen/Qwen3-4B",
    transformer_id: str = "black-forest-labs/flux-2-8b-fp8",
    batch_size: int = 4,
    epochs: int = 20,
    learning_rate: float = 1e-4,
    device: str = "cuda",
    lora_rank: int = 16,
    lora_alpha: int = 32,
    train_proportion: float = 0.9
) -> None:
    
    # 1. Device and Precision Setup
    # The RTX 5090 excels at bfloat16 computation.
    weight_dtype: torch.dtype = torch.bfloat16
    torch_device: torch.device = torch.device(device)
    
    print("Loading tokenizer and text encoder (Qwen 3)...")
    # 2. Load Text Encoder and Tokenizer
    tokenizer: Any = AutoTokenizer.from_pretrained(text_encoder_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    
    # We load the text encoder in bfloat16 but keep it frozen.
    text_encoder: nn.Module = AutoModel.from_pretrained(
        text_encoder_id, 
        dtype=weight_dtype, 
        trust_remote_code=True
    ).to(torch_device)
    text_encoder.eval()
    text_encoder.requires_grad_(False)

    print("Loading VAE...")
    # 3. Load VAE
    vae: nn.Module = AutoencoderKL.from_pretrained(
        vae_id, 
        dtype=weight_dtype
    ).to(torch_device)
    vae.eval()
    vae.requires_grad_(False)

    print("Loading Flux 2.0 Transformer in 8-bit...")
    # 4. Load Flux DiT Backbone in 8-bit
    # We use load_in_8bit to leverage bitsandbytes Q8 quantization natively via Hugging Face
    transformer: nn.Module = FluxTransformer2DModel.from_pretrained(
        transformer_id,
        load_in_8bit=True,
        device_map=device
    )
    transformer.requires_grad_(False) # Freeze base model

    print("Injecting LoRA adapters...")
    # 5. Configure and Apply LoRA
    # We target the attention projections and feed-forward linear layers of the DiT blocks
    lora_config: LoraConfig = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_alpha,
        target_modules=["to_q", "to_k", "to_v", "to_out.0", "proj_mlp", "proj_out"],
        init_lora_weights="gaussian"
    )
    transformer = get_peft_model(transformer, lora_config)
    transformer.train()

    # 6. Optimizer Setup
    # We only pass the parameters that require gradients (the LoRA weights)
    trainable_params: List[nn.Parameter] = [p for p in transformer.parameters() if p.requires_grad]
    optimizer: torch.optim.AdamW = torch.optim.AdamW(
        trainable_params, 
        lr=learning_rate, 
        weight_decay=1e-4
    )

    print("Preparing dataset and dataloader...")
    # 7. Dataset Preparation
    # Split the dataset using random_split
    train_size = int(train_proportion * len(dataset))
    val_size = len(dataset) - train_size
    train_subset, _ = random_split(dataset, [train_size, val_size])
    dataloader: DataLoader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)

    print("Starting Training Loop...")
    # 8. Training Loop
    global_step: int = 0
    
    for epoch in range(epochs):
        epoch_loss: float = 0.0

        for step, (batch_images, batch_captions) in tqdm(enumerate(dataloader), desc=f"Epoch: {epoch}"):
            optimizer.zero_grad()
            
            # Current batch size dynamically extracted
            bsz: int = batch_images.shape[0]
            
            # Move images to device and cast to bf16
            batch_images = batch_images.to(torch_device, dtype=weight_dtype)
            
            # --- A. Encode Images to Latents ---
            # Flux VAE expects inputs in [-1, 1] and outputs latents. We apply the standard scaling factor.
            vae_scaling_factor: float = 0.18215 # Default flux scaling, adjust if Flux 2.0 differs
            with torch.no_grad():
                latents: torch.Tensor = vae.encode(batch_images).latent_dist.sample()
                latents = latents * vae_scaling_factor
            
            text_inputs: Dict[str, torch.Tensor] = tokenizer(
                batch_captions,
                padding="max_length",
                max_length=256,
                truncation=True,
                return_tensors="pt"
            ).to(torch_device)
            
            with torch.no_grad():
                # Extract hidden states from Qwen text encoder
                encoder_outputs: Any = text_encoder(**text_inputs, output_hidden_states=True)
                # Usually, diffusion models use the penultimate hidden state or pooled output
                encoder_hidden_states: torch.Tensor = encoder_outputs.hidden_states[-2] 
                
                # Create a mock pooled projection if the specific Flux pipeline requires it
                # (Flux often concatenates pooled CLIP embeddings and T5/Qwen sequences)
                pooled_projections: torch.Tensor = encoder_hidden_states[:, 0, :] 
            
            # --- C. Flow Matching Setup ---
            # Sample pure noise x_0
            noise: torch.Tensor = torch.randn_like(latents)
            
            # Sample random timesteps t from Uniform(0, 1)
            timesteps: torch.Tensor = torch.rand((bsz,), device=torch_device, dtype=weight_dtype)
            
            # Reshape timesteps for broadcasting: (bsz, 1, 1, 1)
            t_expand: torch.Tensor = timesteps.view(bsz, 1, 1, 1)
            
            # Interpolate between noise (x_0) and data (x_1)
            # x_t = (1 - t) * x_0 + t * x_1
            x_t: torch.Tensor = (1.0 - t_expand) * noise + t_expand * latents
            
            # Target velocity vector v
            # v = x_1 - x_0
            target_velocity: torch.Tensor = latents - noise
            
            # --- D. Forward Pass ---
            # Predict the velocity field
            model_pred: torch.Tensor = transformer(
                hidden_states=x_t,
                timestep=timesteps, # Usually raw float timesteps in DiTs
                encoder_hidden_states=encoder_hidden_states,
                pooled_projections=pooled_projections
            ).sample
            
            # --- E. Loss Calculation and Backprop ---
            # Mean Squared Error between predicted velocity and target velocity
            loss: torch.Tensor = nn.functional.mse_loss(model_pred, target_velocity, reduction="mean")
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            global_step += 1
            
            if global_step % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Step {global_step} | Loss: {loss.item():.4f}")
                
    print("Training complete! Saving LoRA weights...")
    transformer.save_pretrained("./pokemon_flux2_lora")

In [9]:
dataset: PokemonDataset = load_dataset()
train_pokemon_flux_lora(dataset, device="cpu")

Loading tokenizer and text encoder (Qwen 3)...


d:\dev\Archive\ML\lora\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\L.PANNETIER\.cache\huggingface\hub\models--Qwen--Qwen3-4B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:00<00:00,

Loading VAE...


d:\dev\Archive\ML\lora\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


OSError: black-forest-labs/flux-2-vae is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo with `token` or log in with `hf auth login`.